# NyayaSahayak - Legal Domain Classifier
CSE276 Project P14  
Team: Ravnoor Kaur (12505889) & Adarsh Kumar Shukla (12513184)  

This notebook trains a classifier to categorize legal queries and statutes into 5 domains:
- `criminal` (BNS / BNSS)
- `constitutional` (Constitution of India)
- `cyber` (IT Act 2000)
- `consumer` (Consumer Protection Act)
- `evidence` (Bharatiya Sakshya Adhiniyam)

### 1. Load preprocessed legal chunks

In [10]:
import json
from pathlib import Path
import pandas as pd
import numpy as np

# Find dataset file
data_file = Path("data/processed/chunks.jsonl")
if not data_file.exists():
    data_file = Path("../data/processed/chunks.jsonl")
if not data_file.exists():
    data_file = Path("backend/data/processed/chunks.jsonl")

with open(data_file, "r", encoding="utf-8") as f:
    rows = [json.loads(line) for line in f if line.strip()]

df = pd.DataFrame(rows)
print(f"Total legal chunks loaded: {len(df):,}")
df[["act", "section", "title", "category", "year"]].head()

Total legal chunks loaded: 2,419


,act,section,title,category,year
0,"Bharatiya Nyaya Sanhita, 2023",1,"Short title, commencement and application",criminal,2023
1,"Bharatiya Nyaya Sanhita, 2023",2,Definitions.,criminal,2023
2,"Bharatiya Nyaya Sanhita, 2023",3,General explanations,criminal,2023
3,"Bharatiya Nyaya Sanhita, 2023",4,Punishments,criminal,2023
4,"Bharatiya Nyaya Sanhita, 2023",5,Commutation of sentence,criminal,2023


### 2. Check class balance across legal categories

In [11]:
# Criminal law has more sections by nature (BNS + BNSS)
counts = df["category"].value_counts().to_frame(name="Count")
counts["Percentage (%)"] = (counts["Count"] / len(df) * 100).round(2)
counts.index.name = "Category"
counts

,Count,Percentage (%)
Category,,
criminal,1601,66.18
constitutional,493,20.38
evidence,198,8.19
consumer,84,3.47
cyber,43,1.78


### 3. Prepare features and split into Train/Test sets (80-20 stratified)

In [ ]:
from sklearn.model_selection import train_test_split

# Combine title and text for input features
X = [f"{r.get('title', '')} {r.get('text', '')}" for r in rows]
y = [r["category"] for r in rows]

# Stratified split so smaller categories (cyber, consumer) are proportionally represented
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Train size: {len(X_train):,} (80%)")
print(f"Test size:    {len(X_test):,} (20%)")

Train size: 1,935 (80%)
Test size:    484 (20%)


### 4. Create and train pipeline (TF-IDF + Logistic Regression)

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
import time

# unigrams + bigrams, min_df=2 to drop rare noise, balanced weights for class imbalance
clf = make_pipeline(
    TfidfVectorizer(ngram_range=(1, 2), min_df=2, sublinear_tf=True),
    LogisticRegression(max_iter=1000, class_weight="balanced", random_state=42)
)

t0 = time.time()
clf.fit(X_train, y_train)
print(f"Pipeline training finished in {time.time() - t0:.2f} seconds.")
print(f"Classes: {list(clf.classes_)}")
print(f"Vocabulary size: {len(clf.named_steps['tfidfvectorizer'].vocabulary_):,} n-grams")

Pipeline training finished in 0.82 seconds.
Classes: ['constitutional', 'consumer', 'criminal', 'cyber', 'evidence']
Vocabulary size: 18,342 n-grams


### 5. Evaluate on test set (Classification Report)

In [5]:
from sklearn.metrics import classification_report

y_pred = clf.predict(X_test)
report = classification_report(y_test, y_pred, digits=4, output_dict=True)
pd.DataFrame(report).transpose().round(4)

                precision  recall  f1-score  support
constitutional     0.9429  1.0000    0.9706     99.0
consumer           0.9444  1.0000    0.9714     17.0
criminal           0.9936  0.9688    0.9810    320.0
cyber              1.0000  0.8750    0.9333      8.0
evidence           0.8810  0.9250    0.9024     40.0
accuracy           0.9711  0.9711    0.9711    484.0
macro avg          0.9524  0.9537    0.9518    484.0
weighted avg       0.9723  0.9711    0.9713    484.0

### 6. Confusion Matrix

In [6]:
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(y_test, y_pred, labels=clf.classes_)
cm_df = pd.DataFrame(cm, index=clf.classes_, columns=clf.classes_)
cm_df.index.name = "Actual Ground Truth"
cm_df.columns.name = "Predicted"
cm_df

Predicted            constitutional  consumer  criminal  cyber  evidence
Actual Ground Truth                                                 
constitutional                   99         0         0      0         0
consumer                          0        17         0      0         0
criminal                          5         1       310      0         4
cyber                             0         0         0      7         1
evidence                          1         0         2      0        37

### 7. Test on sample user queries

In [7]:
samples = [
    "Can police arrest without warrant in theft cases?",
    "What are fundamental rights guaranteed under Article 21?",
    "Defective laptop delivered by online seller refund complaint",
    "Hacking email account and identity theft online fraud"
]

for s in samples:
    probs = clf.predict_proba([s])[0]
    top_class = clf.classes_[np.argmax(probs)]
    print(f"Query: '{s}'")
    print(f" -> Predicted: {top_class.upper()} ({np.max(probs)*100:.1f}% confidence)\n")

Query: 'Can police arrest without warrant in theft cases?'
 -> Predicted: CRIMINAL (94.6% confidence)

Query: 'What are fundamental rights guaranteed under Article 21?'
 -> Predicted: CONSTITUTIONAL (98.1% confidence)

Query: 'Defective laptop delivered by online seller refund complaint'
 -> Predicted: CONSUMER (96.3% confidence)

Query: 'Hacking email account and identity theft online fraud'
 -> Predicted: CYBER (91.8% confidence)


### 8. Save model for backend API

In [8]:
import joblib

out_path = Path("data/topic_clf.joblib")
if not out_path.parent.exists():
    out_path = Path("../data/topic_clf.joblib")
if not out_path.parent.exists():
    out_path = Path("backend/data/topic_clf.joblib")

joblib.dump(clf, out_path)
print(f"✓ Model saved to: {out_path}")
print("Ready for FastAPI backend.")

✓ Model saved to: data/topic_clf.joblib
Ready for FastAPI backend.
